# RSNA Knee — Training

Thin shell: pulls `knee` from GitHub at a pinned commit, trains on the pre-mounted
competition data, exports checkpoints to `/kaggle/working`.

Current experiment: **E009-laterality-sagt1** — the two input-level deltas vs the
documented 0.92-tier pipelines, stacked on the (now working) E008 recipe:
(1) **laterality normalization** — volumes are mirrored onto a canonical
right-knee, patient-left-increasing frame (side derived from the image center in
patient coordinates; coronal/axial flip horizontally, sagittal reverses slice
order), so medial/lateral anatomy stops swapping sides between knees. Targets our
four weakest labels (MCL 0.722, Medial Meniscus 0.746, ACL 0.750, Lateral
Meniscus 0.766 in E008) — all side-specific thin structures. (2) **Sagittal T1
fourth sequence slot** — marrow/bone contrast for OA/Fracture/Contusion; every
documented strong pipeline runs 4 slots. Same fixed 90/10 holdout as E008 →
paired read against its 0.785.

GPU cost (single arm): decode 4 slots ~70 min + unified fine-tune at ~32 bag
items/study ~2-2.5h → **total ~3-3.5 T4 hours**. Requires `WANDB_API_KEY`,
`knee-labels`, internet.

In [ ]:
# Pin a commit so every checkpoint traces to exact code. Training notebooks have internet.
# --no-deps everywhere: Kaggle's image already ships torch/timm/sklearn/numpy compiled
# together; letting pip resolve our pins upgrades numpy and breaks the whole stack.
COMMIT = "5290655"  # main @ PR #25 squash: E009 laterality normalization + Sagittal T1 slot
%pip install -q --no-deps "git+https://github.com/Josie29/capstone-rsna-knee@{COMMIT}#egg=knee"
%pip install -q --no-deps pydicom pylibjpeg pylibjpeg-libjpeg pylibjpeg-openjpeg pylibjpeg-rle

import numpy  # fail fast if the image stack is broken or missing
import timm
import torch

print("numpy", numpy.__version__, "| torch", torch.__version__, "| timm", timm.__version__)

from knee.series import SeriesType
from knee.train_blended import BLENDED_LABEL_SOURCE

In [ ]:
# Competition data (DICOMs + series metadata) is pre-mounted; blended soft labels
# (with the per-cell __weight companions for E006a) via the knee-labels dataset;
# the E005 winner's feature bank + checkpoints via knee-e005-artifacts.
from pathlib import Path

from knee.data import load_blended_labels, weight_matrix

SLUG = "rsna-knee-abnormality-detection"
candidates = [Path("/kaggle/input/competitions") / SLUG, Path("/kaggle/input") / SLUG]
COMP_ROOT = next((p for p in candidates if (p / "train.csv").exists()), None)
if COMP_ROOT is None:
    listing = {str(d): [c.name for c in d.iterdir()] for d in Path("/kaggle/input").iterdir() if d.is_dir()}
    raise FileNotFoundError(f"competition data not found; mounts: {listing}")
print("competition root:", COMP_ROOT)


def find_input(name: str, filename: str) -> Path:
    bases = [Path("/kaggle/input") / name, Path("/kaggle/input/datasets/josiemachalek") / name]
    for base in bases:
        if (base / filename).exists():
            return base / filename
    listing = {str(d): [c.name for c in d.iterdir()] for d in Path("/kaggle/input").iterdir() if d.is_dir()}
    raise FileNotFoundError(f"{name}/{filename} not found; mounts: {listing}")


labels = load_blended_labels(find_input("knee-labels", "blended_labels_v1.csv"), include_weights=True)
weights = weight_matrix(labels)
print(f"blended labels: {len(labels)} studies; weights {weights.shape}")

In [ ]:
# Metrics/hyperparams only -- no report text, no StudyInstanceUIDs (rule 2.4.b).
# Best-effort: the run proceeds without wandb if the secret isn't configured.
run = None
try:
    import wandb
    from kaggle_secrets import UserSecretsClient

    wandb.login(key=UserSecretsClient().get_secret("WANDB_API_KEY"))
    run = wandb.init(
        project="rsna-knee",
        config={"commit": COMMIT, "label_source": BLENDED_LABEL_SOURCE, "n_label_studies": len(labels)},
    )
except Exception as exc:  # noqa: BLE001 — telemetry must never kill a training run
    print(f"wandb disabled: {exc}")

In [ ]:
# E009 config = E008's trainer (which finally fine-tunes cleanly) + the two input
# fixes. The 4th slot is APPENDED — plane-embedding indices are positional, so
# order changes would silently remap planes on any warm start.
from knee.model import DEFAULT_BACKBONE

SERIES_TYPES = [
    SeriesType.SAGITTAL_FLUID,
    SeriesType.CORONAL_FLUID,
    SeriesType.AXIAL_FLUID,
    SeriesType.SAGITTAL_NONFLUID,  # E009: the T1 slot
]
BACKBONE = DEFAULT_BACKBONE
INPUT_SIZE = 224
CROP_MM = 140.0
USE_WEIGHTS = True
CANONICALIZE_LATERALITY = True  # E009: canonical right-knee frame, baked into the cache
E009_CONFIG = {"n_anchors": 8, "anchor_window": (0.1, 0.9), "frozen_epochs": 4, "epochs": 18}

CHECKPOINT_DIR = Path("/kaggle/working")
# Fresh dir name: cache files carry no geometry version, so a laterality change
# over an old cache would silently mix mirrored and raw frames.
CACHE_DIR = Path("/tmp/pixel_cache_e009")

In [ ]:
# Fixed 90/10 split — the fine-tune-era regime marker. Same seed always, so every
# fine-tune era experiment shares the split and stays comparable.
import numpy as np

from knee.cv import stratified_holdout
from knee.labels import LABEL_COLUMNS

label_matrix = labels[list(LABEL_COLUMNS)].to_numpy(dtype=np.float32)
val_mask = stratified_holdout(label_matrix, val_fraction=0.1, seed=0)
print(f"split: {int((~val_mask).sum())} train / {int(val_mask.sum())} val")

In [ ]:
# E009: pixel cache (~70 min for 4 slots), then ONE unified fine-tune (~2-2.5h at
# 8 anchors x up to 4 planes = up to 32 images per study). Sanity-read the
# laterality tally before trusting the run: expect roughly half left_mirrored,
# half right, ~97% resolved overall, and only a handful ambiguous (the bilateral
# series) or no_geometry — a skewed tally means the geometry read is broken.
from knee.finetune import FinetuneConfig, build_pixel_cache, finetune_unified
from knee.model import MultiPlaneModel

cache = build_pixel_cache(
    COMP_ROOT, labels, CACHE_DIR, series_types=SERIES_TYPES,
    input_size=INPUT_SIZE, crop_mm=CROP_MM,
    canonicalize_laterality=CANONICALIZE_LATERALITY,
)
print("cache coverage:", {t.value: n for t, n in cache.coverage.items()})
print("laterality:", {outcome.value: n for outcome, n in cache.laterality.items()})

model = MultiPlaneModel(BACKBONE, SERIES_TYPES)
result = finetune_unified(
    CACHE_DIR, label_matrix, val_mask,
    model=model,
    out_path=CHECKPOINT_DIR / "e009_unified.pt",
    config=FinetuneConfig(**E009_CONFIG),
    input_size=INPUT_SIZE, crop_mm=CROP_MM,
    cell_weights=weights if USE_WEIGHTS else None,
    label_source=BLENDED_LABEL_SOURCE,
    laterality_normalized=CANONICALIZE_LATERALITY,
)
print(f"E009 unified: best val macro {result.best_val_macro_auc:.3f} (epoch {result.best_epoch + 1})")
print({label: round(auc, 3) for label, auc in result.val_auc_per_label.items()})

In [ ]:
# e009_unified.pt persists as notebook output. Submission bar (team policy
# 2026-09-03): >=0.80 holdout macro justifies the publish->infer->submit path;
# below that, record the numbers and bank the learning. The checkpoint stamps
# laterality_normalized, so inference mirrors its inputs automatically.
import math

if run is not None:
    run.config.update({
        "backbone": BACKBONE, "input_size": INPUT_SIZE, "crop_mm": CROP_MM,
        "tier_weighted": USE_WEIGHTS, "model_kind": "multiplane",
        "n_planes": len(SERIES_TYPES), "laterality_normalized": CANONICALIZE_LATERALITY,
        **E009_CONFIG,
    })
    wandb.log({
        "holdout/unified_macro": result.best_val_macro_auc,
        **{f"holdout/unified/{label}": auc for label, auc in result.val_auc_per_label.items() if not math.isnan(auc)},
    })
    run.finish()